In [ ]:
%%sql -r dataframe_1
CREATE DATABASE HOTEL_DB;

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE FILE FORMAT FF_CSV
    TYPE = 'CSV'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1
    NULL_IF = ('NULL','null','')

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE STAGE STG_HOTEL_BOOKINGS
    FILE_FORMAT = FF_CSV;

In [ ]:
%%sql -r dataframe_4
CREATE TABLE BRONZE_HOTEL_BOOKINGS (
booking_id STRING,
hotel_id STRING,
hotel_city STRING,
customer_id STRING,
customer_name STRING,
customer_email STRING,
check_in_date STRING,
check_out_date STRING,
room_type STRING,
num_guests STRING,
total_amount STRING,
currency STRING,
booking_status STRING
)

In [ ]:
%%sql -r dataframe_5
COPY INTO BRONZE_HOTEL_BOOKINGS
FROM @STG_HOTEL_BOOKINGS
FILE_FORMAT = (FORMAT_NAME = FF_CSV)
ON_ERROR = 'Continue';

In [ ]:
%%sql -r dataframe_6
SELECT *
FROM BRONZE_HOTEL_BOOKINGS
LIMIT 10;

In [ ]:
%%sql -r dataframe_7
CREATE TABLE SILVER_HOTEL_BOOKINGS (
booking_id VARCHAR,
hotel_id VARCHAR,
hotel_city VARCHAR,
customer_id VARCHAR,
customer_name VARCHAR,
customer_email VARCHAR,
check_in_date DATE,
check_out_date DATE,
room_type VARCHAR,
num_guests INTEGER,
total_amount FLOAT,
currency VARCHAR,
booking_status VARCHAR
);

In [ ]:
%%sql -r dataframe_8
SELECT customer_email
FROM BRONZE_HOTEL_BOOKINGS
WHERE NOT (customer_email LIKE '%@%.%')
      OR customer_email IS NULL;

In [ ]:
%%sql -r dataframe_9
SELECT total_amount
FROM BRONZE_HOTEL_BOOKINGS
WHERE TRY_TO_NUMBER(total_amount) < 0;

In [ ]:
%%sql -r dataframe_10
SELECT check_in_date, check_out_date
FROM BRONZE_HOTEL_BOOKINGS
WHERE TRY_TO_DATE(check_in_date) > TRY_TO_DATE(check_out_date);

In [ ]:
%%sql -r dataframe_11
SELECT DISTINCT booking_status
FROM BRONZE_HOTEL_BOOKINGS;


In [ ]:
%%sql -r dataframe_12
INSERT INTO SILVER_HOTEL_BOOKINGS
SELECT
    booking_id,
    hotel_id,
    
    INITCAP(TRIM(hotel_city)) AS hotel_city,
    
    customer_id,
    
    INITCAP(TRIM(customer_name)) AS cutomer_name,
    
    CASE
       WHEN customer_email LIKE '%@%.%' THEN LOWER(TRIM(customer_email))
       ELSE NULL
    END AS customer_email,
    
    TRY_TO_DATE(NULLIF(check_in_date, '')) AS check_in_date,
    
    TRY_TO_DATE(NULLIF(check_out_date, '')) AS check_out_date,
    
    room_type ,
    num_guests ,
    
    ABS(TRY_TO_NUMBER(total_amount)) AS total_amount,
    
    currency,
    
    CASE
    WHEN LOWER(booking_status) IN ('confirmeeed','confirmed') THEN 'Confirmed'
    ELSE booking_status
    END AS booking_status
FROM BRONZE_HOTEL_BOOKINGS
WHERE 
   TRY_TO_DATE(check_in_date) IS NOT NULL
   AND
   TRY_TO_DATE(check_out_date) IS NOT NULL
   AND 
   TRY_TO_DATE(check_in_date) <= TRY_TO_DATE(check_out_date)


In [ ]:
%%sql -r dataframe_13
SELECT * 
FROM SILVER_HOTEL_BOOKINGS
LIMIT 50;

In [ ]:
%%sql -r dataframe_14
CREATE OR REPLACE TABLE GOLD_AGG_DAILY_BOOKING AS
SELECT
    check_in_date AS DATE,
    COUNT(*) AS Total_Booking,
    SUM(total_amount) AS Total_Revenue
FROM silver_hotel_bookings
GROUP BY check_in_date
ORDER BY DATE;

In [ ]:
%%sql -r dataframe_15
CREATE OR REPLACE TABLE GOLD_AGG_HOTEL_CITY_SALES AS
SELECT
    hotel_city,
    SUM(total_amount) AS Total_Revenue
FROM SILVER_HOTEL_BOOKINGS
GROUP BY hotel_city
ORDER BY Total_Revenue DESC;

In [ ]:
%%sql -r dataframe_16
CREATE TABLE GOLD_BOOKING_CLEAN AS
SELECT
    booking_id,
    hotel_id,
    hotel_city,
    customer_id,
    customer_name,
    customer_email,
    check_in_date,
    check_out_date,
    room_type,
    num_guests,
    total_amount,
    currency,
    booking_status
FROM SILVER_HOTEL_BOOKINGS;

In [ ]:
%%sql -r dataframe_17
SELECT * FROM GOLD_AGG_DAILY_BOOKING LIMIT 30;

In [ ]:
%%sql -r dataframe_18
SELECT * FROM GOLD_AGG_HOTEL_CITY_SALES LIMIT 30;